# CME Futures: Tabular Deep Learning

TabM applies a parameter-efficient neural ensemble to the same point-in-time feature rows used by
the linear and gradient-boosting families. The declared configurations vary model capacity while
retaining the walk-forward fold and label contracts from `05_evaluation`.

The shared runner publishes every declared epoch checkpoint with its fitted weights and exact
validation coverage. The equal-weight validation backtest in `13_backtest` evaluates all
checkpoints and selects by Sharpe.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Fit the declared CME futures TabM population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# The population hash this run replaces, read from the registry and set by a person. A
# first population takes None; a re-run whose membership has changed is refused without
# the hash it supersedes, and the refusal names the value required.
SUPERSEDES_POPULATION: str | None = None

## Declared requests

Both configured return horizons enter the same visible request table. Preview epoch or fold limits
must be passed through `PREVIEW_REDUCTIONS`, which changes identity and excludes the output from
the canonical catalog.

**TabM runs on the GPU, and the request says so rather than inheriting it.** With no override the
shared adapter falls back to a literal `"cuda"` written in `case_studies/utils/tabular_dl.py`, and
`resolve_torch_device` raises `CUDA was requested but is unavailable` rather than quietly moving
the fit to the CPU. A CUDA device is therefore a hard requirement of this population, declared two
layers below the notebook: without one these configurations cannot be reproduced at all. Naming it
in the request puts that requirement where a reader meets it. The resolved specification hash is
the same with the override as without, so this states what the published run already did.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog("tabular_dl", labels=ALL_LABELS)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": "cuda"},
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,date,date,i64,str,str
"""tabular_dl""","""fwd_ret_21d""","""tabm_l""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,8,"""canonical""","""c626cf46688a"""
"""tabular_dl""","""fwd_ret_21d""","""tabm_m""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,8,"""canonical""","""5ccff0bd685d"""
"""tabular_dl""","""fwd_ret_21d""","""tabm_s""","""regression""",69,30,37782,5,2019-01-03,2023-11-29,8,"""canonical""","""d575ade23232"""
"""tabular_dl""","""fwd_ret_5d""","""tabm_l""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,8,"""canonical""","""f1ba0c34a929"""
"""tabular_dl""","""fwd_ret_5d""","""tabm_m""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,8,"""canonical""","""b8688e1ee906"""
"""tabular_dl""","""fwd_ret_5d""","""tabm_s""","""regression""",69,30,38262,5,2019-01-03,2023-12-21,8,"""canonical""","""0f5b1db30a12"""


## Execute and validate

Fold-scoped preprocessing, seeded training, fitted-state persistence, checkpoint membership, and
prediction eligibility are enforced by the shared TabM adapter.

In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name="cme_futures-tabular_dl-validation-v1",
        resolved_requests=resolved,
        supersedes=SUPERSEDES_POPULATION,
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

Preparing and releasing folds...


  Fold 0: train=60,680  val=7,518


      epoch  25/200: loss=0.000956, IC=-0.0483


      epoch  50/200: loss=0.000800, IC=-0.0523


      epoch  75/200: loss=0.000714, IC=-0.0404


...

      epoch 175/200: loss=0.000343, IC=+0.0467


      epoch 200/200: loss=0.000342, IC=+0.0465


    Fold 4: best_ep=25, IC=+0.0490 (20.9s)


    → best_epoch=25, IC=-0.0264 (99.5s)



  Best: c626cf46688a @ epoch 25 (IC=-0.0264)


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "config_name", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("TabM execution returned a partial prediction")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""tabular_dl""","""fwd_ret_21d""","""tabm_l""","""epoch""",25,"""canonical""",true,"""c626cf46688a""","""418039bbf729"""
"""tabular_dl""","""fwd_ret_21d""","""tabm_l""","""epoch""",50,"""canonical""",true,"""c626cf46688a""","""801d69d13dff"""
"""tabular_dl""","""fwd_ret_21d""","""tabm_l""","""epoch""",75,"""canonical""",true,"""c626cf46688a""","""1f62cf8d02e4"""
"""tabular_dl""","""fwd_ret_21d""","""tabm_l""","""epoch""",100,"""canonical""",true,"""c626cf46688a""","""f07808693cc8"""
"""tabular_dl""","""fwd_ret_21d""","""tabm_l""","""epoch""",125,"""canonical""",true,"""c626cf46688a""","""1678f2cc1d0c"""
…,…,…,…,…,…,…,…,…
"""tabular_dl""","""fwd_ret_5d""","""tabm_s""","""epoch""",100,"""canonical""",true,"""0f5b1db30a12""","""329899b680cc"""
"""tabular_dl""","""fwd_ret_5d""","""tabm_s""","""epoch""",125,"""canonical""",true,"""0f5b1db30a12""","""53aa5ef6c84c"""
"""tabular_dl""","""fwd_ret_5d""","""tabm_s""","""epoch""",150,"""canonical""",true,"""0f5b1db30a12""","""484c6fae9092"""
